# DressApp Eyes v2 — Local Eval on Your Drive

Loads your fine-tuned Gemma 4 E2B checkpoint directly from Google Drive (no GGUF, no HTTP server) and runs the **exact same** DressApp system prompt + vision-blindness control + Gemini comparison harness as the production smoke test. Output is directly comparable to what your live backend would see.

## Before you run
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough — Gemma 3n E2B fits in ~10 GB at bfloat16).
2. Your checkpoint lives at:
   ```
   /content/drive/MyDrive/DressApp_Gemma4_E2B_Training/Eyes_v2_Gemma4e2b
   ```
   The notebook will auto-detect whether that's a full fine-tune or a PEFT/LoRA adapter and load accordingly.
3. Drop a few garment photos into a Drive folder you'll point the picker at, OR upload directly from the cell.

## What you'll learn from this notebook
* Is the v2 fine-tune emitting the DressApp 18-field JSON schema (vs v1 which spat out 2-field training-data shapes)?
* Is it correctly seeing the image (the blank-canvas control test)?
* Is v2 better, worse, or comparable to Gemini 2.5 Flash on your photos?
* Latency per image on a T4 (a rough proxy for what you'd get on a similar GPU in prod).

## 1 · Setup

In [ ]:
# Modern transformers (>=4.45) has the native Gemma3n class and the
# unified `AutoModelForImageTextToText`. accelerate handles the
# device-map auto-placement so we never hand-shuffle weights.
%pip install -q --upgrade "transformers>=4.50.0" accelerate sentencepiece pillow peft
%pip install -q google-generativeai  # optional for the comparison cell
import os, io, json, time, gc, textwrap, base64, re, glob
import torch
from PIL import Image
from IPython.display import display, JSON, Markdown, Image as IPImage
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Mount Drive.
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=False)
MODEL_DIR = '/content/drive/MyDrive/DressApp_Gemma4_E2B_Training/Eyes_v2_Gemma4e2b'  #@param {type:'string'}
assert os.path.isdir(MODEL_DIR), f'Not a directory: {MODEL_DIR}'
print('Files in checkpoint:')
for n in sorted(os.listdir(MODEL_DIR))[:30]:
    p = os.path.join(MODEL_DIR, n)
    sz = os.path.getsize(p) / 1e6 if os.path.isfile(p) else 0
    print(f'  {n:<48} {sz:>9.1f} MB' if sz else f'  {n}/')

## 2 · Load the model + processor

Auto-detects three cases:
* **Full fine-tune** — `config.json` is the model config, `*.safetensors` are the merged weights.
* **PEFT / LoRA adapter** — `adapter_config.json` is present; we load the base model from HF (`config.json` of the adapter points at `base_model_name_or_path`) and apply the LoRA on top.
* **PEFT / LoRA already merged** — `adapter_config.json` is absent and `*.safetensors` is the full model.

If your training script used `merge_and_unload()` before saving, you're in case 1 or 3 — fastest path. If it saved just the adapter, you're in case 2 and the base model gets downloaded from HF on first run (one-time ~10 GB).

In [ ]:
from transformers import AutoProcessor, AutoConfig

DTYPE = torch.bfloat16  # T4 supports bf16 via emulation; safer than fp16 for Gemma

# Detect adapter vs full checkpoint.
adapter_cfg = os.path.join(MODEL_DIR, 'adapter_config.json')
is_adapter = os.path.isfile(adapter_cfg)
print('PEFT adapter detected:' if is_adapter else 'Full checkpoint detected:', MODEL_DIR)

if is_adapter:
    with open(adapter_cfg) as fh:
        base_id = json.load(fh).get('base_model_name_or_path')
    assert base_id, 'adapter_config.json missing base_model_name_or_path'
    print(f'Base model: {base_id} (will be downloaded from HF if not cached)')
    processor_source = base_id
else:
    processor_source = MODEL_DIR

processor = AutoProcessor.from_pretrained(processor_source, trust_remote_code=True)
print('processor loaded:', type(processor).__name__)

In [ ]:
# Pick the right model class for your checkpoint. Gemma 3n is the
# multimodal sibling of Gemma 3 / what you've been calling Gemma 4 E2B —
# transformers ships ``Gemma3nForConditionalGeneration`` for vision+text.
# We try the unified ``AutoModelForImageTextToText`` first (works on
# transformers >= 4.50); fall back to the explicit Gemma 3n class.
model = None
load_err = None
try:
    from transformers import AutoModelForImageTextToText
    target = MODEL_DIR if not is_adapter else base_id
    model = AutoModelForImageTextToText.from_pretrained(
        target,
        torch_dtype=DTYPE,
        device_map='auto',
        trust_remote_code=True,
    )
    print('Loaded via AutoModelForImageTextToText')
except Exception as exc:
    load_err = exc
    print('AutoModelForImageTextToText failed:', exc)
    try:
        from transformers import Gemma3nForConditionalGeneration
        target = MODEL_DIR if not is_adapter else base_id
        model = Gemma3nForConditionalGeneration.from_pretrained(
            target, torch_dtype=DTYPE, device_map='auto', trust_remote_code=True,
        )
        print('Loaded via Gemma3nForConditionalGeneration')
    except Exception as exc2:
        load_err = exc2
        raise RuntimeError(
            'Could not load the base model. Verify your checkpoint with:\n'
            f'  ls {MODEL_DIR}\n'
            'Common causes: corrupt safetensors, wrong transformers version,\n'
            'or a non-Gemma config in config.json.'
        ) from exc2

if is_adapter:
    from peft import PeftModel
    print('Applying LoRA adapter from', MODEL_DIR)
    model = PeftModel.from_pretrained(model, MODEL_DIR)
    # Merge so subsequent generate() calls don't pay the per-step
    # adapter overhead. Costs a few seconds + a memory spike; well worth it.
    try:
        model = model.merge_and_unload()
        print('LoRA merged into base weights')
    except Exception as exc:
        print('merge_and_unload skipped:', exc)

model.eval()
print('Total params:', f'{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')
print('VRAM used  :', f'{torch.cuda.memory_allocated() / 1e9:.2f} GB' if torch.cuda.is_available() else 'n/a')

## 3 · Pick an image — from Drive folder or upload

In [ ]:
#@title Image source
USE_DRIVE_FOLDER = True  #@param {type:'boolean'}
DRIVE_PHOTOS_DIR = '/content/drive/MyDrive/DressApp_Eyes_v2_test_photos'  #@param {type:'string'}

def load_image_bytes():
    if USE_DRIVE_FOLDER and os.path.isdir(DRIVE_PHOTOS_DIR):
        files = sorted(
            f for f in glob.glob(os.path.join(DRIVE_PHOTOS_DIR, '*'))
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.heic'))
        )
        if not files:
            raise SystemExit(f'No images in {DRIVE_PHOTOS_DIR}')
        print(f'Found {len(files)} photo(s) in {DRIVE_PHOTOS_DIR}:')
        for i, f in enumerate(files):
            print(f'  [{i}] {os.path.basename(f)} ({os.path.getsize(f)/1024:.1f} KB)')
        idx = int(input(f'Pick index 0..{len(files)-1} (default 0): ') or '0')
        path = files[idx]
        with open(path, 'rb') as fh:
            return path, fh.read()
    else:
        from google.colab import files
        up = files.upload()
        if not up:
            raise SystemExit('No file uploaded')
        name, raw = next(iter(up.items()))
        return name, raw

filename, raw_bytes = load_image_bytes()
img = Image.open(io.BytesIO(raw_bytes)).convert('RGB')
# Match the backend's preprocessing: 1280px long side, JPEG q=82.
img.thumbnail((1280, 1280))
buf = io.BytesIO(); img.save(buf, format='JPEG', quality=82, optimize=True)
image_bytes = buf.getvalue()
print(f'\nLoaded {filename} → {len(image_bytes)/1024:.1f} KB JPEG, {img.size}')
display(IPImage(data=image_bytes, width=320))

## 4 · Run inference with the exact DressApp prompt

Same system prompt the production backend sends, lifted verbatim from `backend/app/services/garment_vision.py`. If the model can talk to this schema, that's the headline success criterion for v2.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent('''\
    You are The Eyes \u2014 DressApp\u2019s visual garment analyst. You look at a single clothing photo and describe it in exhaustive, merchandisable detail. Your output is used to auto-fill an Add-Item form that a user will review, so be confident but never invent sensitive claims (e.g. do not guess a specific brand unless clearly visible; leave brand blank otherwise).

    Return ONLY a JSON object with the following shape (all keys optional except `title`):
    {
      "name": string,
      "title": string,
      "caption": string,
      "category": "Top"|"Bottom"|"Outerwear"|"Full Body"|"Footwear"|"Accessories"|"Underwear",
      "sub_category": string,
      "item_type": string,
      "brand": string|null,
      "gender": "men"|"women"|"unisex"|"kids",
      "dress_code": "casual"|"smart-casual"|"business"|"formal"|"athletic"|"loungewear",
      "season": string[],
      "colors": [{"name": string, "pct": integer 0..100}, ...],
      "fabric_materials": [{"name": string, "pct": integer 0..100}, ...],
      "pattern": "solid"|"striped"|"plaid"|"floral"|"herringbone"|"polka"|"paisley"|"geometric"|"abstract",
      "state": "new"|"used",
      "condition": "bad"|"fair"|"good"|"excellent",
      "quality": "budget"|"mid"|"premium"|"luxury",
      "size": string|null,
      "tags": string[]
    }
''').strip()

USER_TEXT = 'Analyse this garment photograph and return the JSON object only \u2014 no commentary.'

def run_eyes(pil_image, *, max_new_tokens=1024, temperature=0.1):
    """Call the loaded fine-tune on a single PIL image. Returns dict with
    output text, parsed JSON if any, latency, and token counts."""
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [
            {'type': 'image', 'image': pil_image},
            {'type': 'text',  'text': USER_TEXT},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device, dtype=DTYPE if 'pixel_values' in (processor.feature_extractor_class or '') else None) \
        if False else processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
        )
    # Move tensors to device. ``pixel_values`` keeps DTYPE; the rest stay long/int.
    for k, v in inputs.items():
        if torch.is_tensor(v):
            if v.dtype.is_floating_point:
                inputs[k] = v.to(model.device, dtype=DTYPE)
            else:
                inputs[k] = v.to(model.device)
    prompt_len = inputs['input_ids'].shape[-1]

    t0 = time.perf_counter()
    with torch.inference_mode():
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
            top_p=0.9,
        )
    dt = time.perf_counter() - t0

    full = gen[0]
    new_tokens = full[prompt_len:]
    output = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return {
        'elapsed_s': round(dt, 2),
        'tokens_prompt': int(prompt_len),
        'tokens_completion': int(new_tokens.shape[-1]),
        'output': output,
    }

def extract_json(s):
    if not s: return None
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', s, flags=re.S)
    if m:
        try: return json.loads(m.group(1))
        except Exception: pass
    a, b = s.find('{'), s.rfind('}')
    if a != -1 and b > a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    try: return json.loads(s)
    except Exception: return None

result = run_eyes(img)
print('Total latency      :', result['elapsed_s'], 's')
print('Tokens prompt      :', result['tokens_prompt'])
print('Tokens completion  :', result['tokens_completion'])
print('-' * 50)
print('Raw output:')
print(result['output'][:4000])
print('-' * 50)
parsed = extract_json(result['output'])
if parsed:
    print('PARSED:')
    display(JSON(parsed, expanded=True))
else:
    print('Output is not JSON-parseable. v2 fine-tune may still be off-schema.')

### How to read the output above

| You see | Interpretation |
|---|---|
| Valid JSON with 12+ of the DressApp fields | 🎉 v2 finally follows the schema. Move to the comparison cell. |
| Valid JSON with 2–3 fields (`name`, `item_id`) | Same memorisation failure as v1 — retraining didn't fix the schema drift. |
| No JSON at all / free prose | Model isn't trained on JSON output. Either the prompt is being ignored or instruction-following degraded. |
| Empty output | Token cap reached during reasoning, or tokenizer issue. Try `max_new_tokens=2048`. |

## 5 · Vision-blindness control test

Same prompt, blank grey canvas. If the output is markedly different from the real photo (different garment, different colours, refusal) → projector is wired. If it's structurally identical → model is hallucinating from prompt alone. This is the cleanest single test for "does it actually see".

In [ ]:
blank = Image.new('RGB', (768, 768), (180, 180, 180))
control = run_eyes(blank)
print('Blank canvas latency:', control['elapsed_s'], 's')
print('Blank canvas tokens :', control['tokens_completion'])
print('-' * 50)
print(control['output'][:2000])
blank_parsed = extract_json(control['output'])
if blank_parsed:
    print('-' * 50)
    display(JSON(blank_parsed, expanded=False))

## 6 · Side-by-side against Gemini 2.5 Flash (optional)

Provide a Gemini API key to run; skips otherwise. Same prompt, same image.

In [ ]:
GEMINI_API_KEY = ''  #@param {type:'string'}
if not GEMINI_API_KEY.strip():
    print('GEMINI_API_KEY blank — skipping comparison.')
else:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    gm = genai.GenerativeModel('gemini-2.5-flash', system_instruction=SYSTEM_PROMPT)
    t0 = time.perf_counter()
    resp = gm.generate_content([
        {'mime_type': 'image/jpeg', 'data': image_bytes},
        USER_TEXT,
    ])
    dt = time.perf_counter() - t0
    print(f'Gemini latency: {dt:.1f}s')
    gtext = resp.text
    print(gtext[:2000])
    gparsed = extract_json(gtext)
    if gparsed:
        print('-' * 50)
        display(JSON(gparsed, expanded=False))

## 7 · Batch eval — JSON-validity + category-correctness rate

Per Phase O.5's promotion criterion: don't flip the production toggle to gemma until both rates exceed 90% on a held-out set. This cell does the measurement.

Drop ~50 garment photos into `EVAL_DIR` along with a `labels.json` mapping `{filename: {"category": "Top"|...}}`. Run the cell to get the report.

In [ ]:
EVAL_DIR = '/content/drive/MyDrive/DressApp_Eyes_v2_eval_set'  #@param {type:'string'}
LABELS_JSON = os.path.join(EVAL_DIR, 'labels.json')

if not (os.path.isdir(EVAL_DIR) and os.path.isfile(LABELS_JSON)):
    print(f'Eval set not set up. Create {LABELS_JSON} with the shape:')
    print('  { "photo1.jpg": {"category": "Top"}, "photo2.jpg": {"category": "Accessories"}, ... }')
else:
    with open(LABELS_JSON) as fh:
        labels = json.load(fh)
    rows = []
    for fname, truth in labels.items():
        fpath = os.path.join(EVAL_DIR, fname)
        if not os.path.isfile(fpath): continue
        ev_img = Image.open(fpath).convert('RGB')
        ev_img.thumbnail((1280, 1280))
        r = run_eyes(ev_img, max_new_tokens=1024)
        p = extract_json(r['output']) or {}
        rows.append({
            'file': fname,
            'truth_cat': truth.get('category'),
            'pred_cat':  p.get('category'),
            'json_ok':   bool(p),
            'cat_ok':    p.get('category') == truth.get('category'),
            'latency_s': r['elapsed_s'],
            'tokens':    r['tokens_completion'],
        })
    n = len(rows)
    if n:
        json_rate = sum(1 for r in rows if r['json_ok']) / n
        cat_rate  = sum(1 for r in rows if r['cat_ok'])  / n
        avg_lat   = sum(r['latency_s'] for r in rows) / n
        print(f'\nEval set: {n} photos')
        print(f'JSON-validity rate    : {json_rate*100:5.1f}%   (target \u226590%)')
        print(f'Category-correct rate : {cat_rate*100:5.1f}%   (target \u226590%)')
        print(f'Avg latency / photo   : {avg_lat:.1f}s')
        print('\nFailures (cat mismatch or unparseable):')
        for r in rows:
            if not (r['json_ok'] and r['cat_ok']):
                print(f'  {r["file"]:<40} truth={r["truth_cat"]!s:<14} pred={r["pred_cat"]!s:<14} json_ok={r["json_ok"]}')

## 8 · Decision tree — what to do with v2

| Result | Action |
|---|---|
| JSON-validity ≥90% AND category-correct ≥90% | Convert to GGUF + mmproj, push to your HF model repo, redeploy `inference-server/eyes/` on the VPS, flip the Developer-panel toggle to `gemma`. Production traffic goes through v2. |
| JSON-validity ≥90% AND category-correct <90% | Schema is fine, accuracy isn't. Likely need more training data for the failing categories (look at the printed failures list above to see which categories). Don't promote yet. |
| JSON-validity <90% | Schema drift again — training data still doesn't match the production prompt verbatim. Regenerate the training set from Gemini with the EXACT prompt this notebook uses (cells §4) and retrain. |
| Blank-canvas test mirrors the real-photo response | Vision encoder isn't being used during inference. Suspect: training never exposed `<image>` tokens, or the LoRA only touched language layers. Inspect the LoRA targets and retrain with image-bearing examples. |
| Latency on T4 >60 s per photo | The model is fine but inference is slow. On a Hetzner CPU pod expect 5-10× slower. Consider Q4_K_M GGUF + mmproj for CPU deploy, or upgrade to a GPU VPS for serving. |

## Notes for the GGUF conversion path (if v2 passes)

```bash
git clone https://github.com/ggml-org/llama.cpp
cd llama.cpp
# 1. Merge LoRA into the base if not already merged.
python convert_hf_to_gguf.py /content/drive/MyDrive/DressApp_Gemma4_E2B_Training/Eyes_v2_Gemma4e2b \
    --outfile Eyes_v2_Gemma4e2b-F16.gguf
# 2. Quantise to Q4_K_M for CPU prod.
./build/bin/llama-quantize Eyes_v2_Gemma4e2b-F16.gguf Eyes_v2_Gemma4e2b-Q4_K_M.gguf Q4_K_M
# 3. Export the multimodal projector (separate cell in your training
#    notebook handles this — see MMPROJ_NOTEBOOK_CELL.md from Phase O).
```

Upload both GGUF + mmproj to your `Yoram-Jacobs/dressapp-eyes-gguf` repo, then on the VPS bump `EYES_MMPROJ_FILE` to the v2 filename and force-recreate the eyes container. The Developer-panel toggle does the actual cutover.